# Deploying AI
## Assignment 1: Evaluating Summaries

In [8]:
import sys
print(sys.executable)


c:\Users\ionam\deploying-ai\deploying-ai-env\Scripts\python.exe


A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [9]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [10]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader(url)
docs = loader.load()

document_text = ""

for doc in docs:
    document_text += doc.page_content + "\n"

print(document_text[:1000])
print(f"\nCharacters loaded: {len(document_text)}")

What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShopOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave this storySave this story“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the positive, from the overpowering to the mysterious, from anarchy to sublimity. The negative seems to lie at the root: etymologists trace the word to “nuisance” and “nausea.” Noise is what drives us mad; it sends the Grinch over the edge at Christmastime. (“Oh, the Noise! Noise! Noise! Noise!”) Noise is the sound of madness itself, the din within our minds. The

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [11]:
import openai

print(openai.__version__)

2.41.1


In [17]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()


class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int = 0
    OutputTokens: int = 0

In [35]:
developer_prompt = """
You are an expert technical writer.

Requirements:
- Extract the Author and Title from the article.
- Explain the relevance of the article to an AI professional in 1 paragraph.
- Generate a concise summary no longer than 1000 tokens.
- Write the summary in a tone consistent with Formal Academic Writing.
"""

user_prompt = f"""
Article:

{document_text}
"""


In [36]:
import os

MODEL = os.getenv("MODEL", "gpt-4o-mini")

response = client.responses.parse(
    model=MODEL,
    input=[
        {
            "role": "developer",
            "content": developer_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ],
    text_format=ArticleSummary,
)

article_summary = response.output_parsed

article_summary.InputTokens = response.usage.input_tokens
article_summary.OutputTokens = response.usage.output_tokens

article_summary 

ArticleSummary(Author='Alex Ross', Title='What Is Noise?', Relevance='This article provides a comprehensive exploration of the concept of noise, relevant to AI professionals as it intersects with various fields, including information theory, human-computer interaction, and sound data processing. Understanding noise, both in the auditory realm and as data interference, is crucial for developing algorithms that can effectively filter, analyze, and interpret signals and information in AI applications, particularly in areas such as natural language processing and machine learning.', Summary='The article delves into the multifaceted nature of noise, discussing its semantic evolution, cultural implications, and the distinction between noise and music. Initially representing an undesirable cacophony, noise encapsulates the chaos of modern life, often regarded through lenses of both discomfort and enjoyment. The discourse shifts towards the personal experience of noise, showcasing its complex 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [37]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel

evaluation_model = GPTModel(
    model=MODEL,
    temperature=0,
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=article_summary.Summary,
)

summarization_metric = SummarizationMetric(
    assessment_questions=[
        "Does the summary accurately capture the main points of the article?",
        "Is the summary concise and well-structured?",
        "Does the summary represent the author's intent and key arguments accurately?",
        "Does the summary provide sufficient context for an AI professional to understand the relevance of the article?",
        "Is the summary free from factual inaccuracies or misinterpretations of the article's content?",
    ],
    model=evaluation_model,
    include_reason=True,
)

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Does the summary flow logically and coherently from one point to the next?",
        "Are there any abrupt transitions or gaps in the summary that hinder understanding?",
        "Does the summary avoid redundancy and repetition, ensuring that each point is presented clearly and succinctly?",
        "Are the ideas presented in a clear and organized manner?",
        "Does the summary effectively connect the main points of the article to provide a cohesive understanding?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=evaluation_model,
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Is the communication style consistent with Formal Academic Writing?",
        "Is the language objective and professional?",
        "Is the vocabulary appropriate for an academic audience?",
        "Does the summary avoid casual or conversational phrasing?",
        "Is the tone consistent throughout the summary?",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=evaluation_model,
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Does the summary avoid harmful or dangerous guidance?",
        "Does the summary avoid discriminatory or offensive language?",
        "Does the summary avoid unsupported or misleading claims?",
        "Are the article's ideas represented responsibly?",
        "Does the summary remain appropriate for a professional audience?",
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model=evaluation_model,
)

C:\Users\ionam\AppData\Local\Temp\ipykernel_29724\633068530.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [38]:
summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

Output()

Output()

Output()

Output()

0.8570772883487077

In [39]:
results = {
    "SummarizationScore": round(summarization_metric.score, 2),
    "SummarizationReason": summarization_metric.reason,

    "CoherenceScore": round(coherence_metric.score, 2),
    "CoherenceReason": coherence_metric.reason,

    "TonalityScore": round(tonality_metric.score, 2),
    "TonalityReason": tonality_metric.reason,

    "SafetyScore": round(safety_metric.score, 2),
    "SafetyReason": safety_metric.reason,
}

results

{'SummarizationScore': 0,
 'SummarizationReason': 'The score is 0.00 because the summary introduces multiple pieces of extra information that are not present in the original text, leading to a significant deviation from the original content.',
 'CoherenceScore': 0.72,
 'CoherenceReason': 'The summary presents a logical flow of ideas, transitioning from the definition of noise to its cultural implications and personal experiences. However, there are some abrupt transitions, particularly towards the end where the discussion of technology and information theory feels somewhat disconnected from the previous points. While the summary avoids redundancy and presents ideas clearly, the lack of a cohesive conclusion limits its overall effectiveness.',
 'TonalityScore': 0.82,
 'TonalityReason': 'The response demonstrates a strong alignment with formal academic writing, using objective and professional language throughout. The vocabulary is appropriate for an academic audience, and the tone remai

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [40]:
def evaluate_summary(summary_text):
    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text,
    )

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    return {
        "SummarizationScore": round(summarization_metric.score, 2),
        "SummarizationReason": summarization_metric.reason,
        "CoherenceScore": round(coherence_metric.score, 2),
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": round(tonality_metric.score, 2),
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": round(safety_metric.score, 2),
        "SafetyReason": safety_metric.reason,
    }

Please, do not forget to add your comments.

In [41]:
enhancement_developer_prompt = """
You are an expert technical writer improving an existing article summary.

Use the original article, the previous summary, and its evaluation feedback.

Requirements:
- Make the revised summary more concise than the previous summary.
- Remove any claims that are unsupported by the original article.
- Preserve the article's essential points.
- Prefer explicit statements from the source material over inferred or implied information.
"""

enhancement_user_prompt = f"""
Original article:

{document_text}

Previous summary:

{article_summary.Summary}

Evaluation results:

{results}
"""

In [42]:
enhanced_response = client.responses.parse(
    model=MODEL,
    input=[
        {
            "role": "developer",
            "content": enhancement_developer_prompt,
        },
        {
            "role": "user",
            "content": enhancement_user_prompt,
        },
    ],
    text_format=ArticleSummary,
)

enhanced_article_summary = enhanced_response.output_parsed

enhanced_article_summary.InputTokens = enhanced_response.usage.input_tokens
enhanced_article_summary.OutputTokens = enhanced_response.usage.output_tokens

enhanced_article_summary

ArticleSummary(Author='Alex Ross', Title='What Is Noise? | The New Yorker', Relevance='Explores the complexities of noise in modern life and its cultural significance.', Summary='The article examines the multifaceted concept of noise, tracing its evolution from a negative term associated with disturbance to a broader cultural phenomenon. It discusses the personal experiences of noise, emphasizing the distinction between noise and music, and how this distinction reflects personal agency and societal control. Additionally, it addresses how noise intertwines with technological advancements and information theory, revealing underlying social dynamics and tensions. Through various historical perspectives, the article illustrates how noise serves both disruptive and artistic purposes, shaping our environment and experiences.', Tone='Academic and analytical', InputTokens=7844, OutputTokens=152)

In [43]:
enhanced_results = evaluate_summary(
    enhanced_article_summary.Summary
)

enhanced_results

Output()

Output()

Output()

Output()

{'SummarizationScore': 0,
 'SummarizationReason': 'The score is 0.00 because the summary introduces extra information about technological advancements and social dynamics that are not present in the original text, leading to a significant deviation from the original content.',
 'CoherenceScore': 0.83,
 'CoherenceReason': 'The summary flows logically and presents the main points of the article in a clear and organized manner. It effectively connects the evolution of the concept of noise with personal experiences and societal implications. However, there are minor gaps in transitions between some ideas, which could hinder understanding slightly, but overall, it avoids redundancy and maintains coherence.',
 'TonalityScore': 0.88,
 'TonalityReason': "The response demonstrates a strong alignment with formal academic writing, using objective and professional language throughout. The vocabulary is appropriate for an academic audience, and the summary avoids casual phrasing. The tone remains c

## Enhancement Results

The enhanced summary didn't produce a better overall result. The Summarization score remained at 0.00, while Coherence, Tonality, and Safety scores were slightly lower than the original.

There was some inconsistency between the metrics. The Summarization metric identified unsupported information, while the Safety evaluation stated that the summary avoided unsupported claims. This is useful but can appear inconsistent, since each metric evaluates a different aspect of the summary.

These controls aren't enough on their own. They can help identify possible weaknesses and support ongoing improvement, but human review is still necessary to verify accuracy, assess interpretation, and resolve conflicting evaluations. 


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
